# MSME Linguistic Decoding RL - Colab Training Notebook

This notebook is a reproducible training pipeline for the OpenEnv hackathon submission.

It includes:
- repo setup,
- baseline and deterministic evaluations,
- GRPO training (`train_grpo.py`) with Unsloth/TRL path,
- artifact generation for judges,
- optional Drive export.

Run all cells top-to-bottom.

In [ ]:
# 1) Clone or update repo, then install
# REPO: Google Colab default is /content/msme_env. On Hugging Face, set e.g.:
#   %env MSME_REPO=/data/msme_env
# before running, or change the default in the line below.
import os
import shutil
import subprocess

REPO = os.environ.get("MSME_REPO", "/content/msme_env")
REPO = os.path.abspath(REPO)
GIT_URL = os.environ.get("MSME_GIT", "https://github.com/di35117/msme_env.git")

def _have_repo(path: str) -> bool:
    return os.path.isfile(os.path.join(path, "train_grpo.py"))

if _have_repo(REPO):
    subprocess.run(["git", "-C", REPO, "pull", "origin", "main"], check=False)
elif os.path.isdir(REPO):
    shutil.rmtree(REPO)
    subprocess.run(["git", "clone", GIT_URL, REPO], check=True)
else:
    os.makedirs(os.path.dirname(REPO) or ".", exist_ok=True)
    subprocess.run(["git", "clone", GIT_URL, REPO], check=True)

%cd {REPO}
!pip install -q -e .
# If Unsloth is not installed, train_grpo falls back to HF. Optional: add --no-unsloth to the training cell.

In [ ]:
# 2) Optional: inspect GPU (for training cell)
!nvidia-smi

In [ ]:
# 3) Baseline + deterministic JSON (required for generate_judge_artifacts + pre_submit_check)
!mkdir -p artifacts
!python scripts/run_baseline_eval.py --episodes 30 --output artifacts/baseline_rewards.json
!python scripts/run_deterministic_eval.py --seed 123 --episodes 5 --max_steps 90 --output artifacts/deterministic_eval.json
# Optional extra report (not required for pre_submit_check)
!python scripts/eval.py --episodes 5 --output artifacts/eval_report.json

In [ ]:
# 4) Main GRPO run: 30 episodes, 90 steps/ep, checkpoint every 2 (matches submission README)
# If Unsloth fails, add:  --no-unsloth
!python train_grpo.py --episodes 30 --max_steps_per_episode 90 --save_every 2 --model Qwen/Qwen2.5-1.5B-Instruct --output_dir msme_rl_run

In [ ]:
# 5) Plots from reward_curve.json → artifacts/ (judge_summary.json, manifest, curves)
!python scripts/generate_judge_artifacts.py --training_json msme_rl_run/reward_curve.json --baseline_json artifacts/baseline_rewards.json --output_dir artifacts
# 6) Before/after RL (needs a finished checkpoint; use latest episode_#### if you stopped early)
!python scripts/inference_before_after.py --base Qwen/Qwen2.5-1.5B-Instruct --trained msme_rl_run/episode_0030 --output-dir msme_rl_run --episodes 5 --max-steps 90
!python scripts/make_hero.py --run-dir msme_rl_run
# 7) Packaging check (expects files under artifacts/ and repo layout)
!python scripts/pre_submit_check.py

In [ ]:
# 8) Optional: copy run + artifacts to Google Drive
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p "/content/drive/MyDrive/msme_env_outputs"
!cp -r artifacts msme_rl_run "/content/drive/MyDrive/msme_env_outputs/"